## Bootstrap / Wilson confidence intervals

Adds 95% CIs for every headline metric and regenerates the four bar-chart figures with error bars. Non-LLM: re-fits the credit/health/fraud models from the bundled data and reads the committed LLM per-case results, so it runs fully offline. Run after notebooks 01-04 (or with the committed `results/tables/` present).

In [1]:
# 05_bootstrap_cis.ipynb
# Confidence intervals for all headline metrics + regeneration of the bar-chart
# figures WITH error bars. Re-fits the credit/health/fraud models from bundled
# data (deterministic, seed=42) and reads the committed LLM per-case results
# (results/tables/llm_explanation_per_case.csv) -- NO OpenAI/API call is made.
# Wilson intervals for proportions; percentile bootstrap (2000 resamples) for
# ratios, AUCs, and means. Writes results/tables/bootstrap_cis.csv. Single cell.

# Bootstrap / Wilson confidence intervals for all headline metrics, plus
# regeneration of the bar-chart figures WITH error bars. Non-LLM: re-fits the
# credit/health/fraud models from bundled data (deterministic, seed=42) and
# reads the committed LLM per-case results (no API call).
import json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.model_selection import cross_val_predict, StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
warnings.filterwarnings("ignore")

def find_root():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c/"results").is_dir() and (c/"notebooks").is_dir():
            return c
    return p.parent if p.name=="notebooks" else p
ROOT = find_root()
DATA, FIG, TAB = ROOT/"data", ROOT/"results"/"figures", ROOT/"results"/"tables"
SEED = 42; B = 2000; rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"figure.dpi":120,"savefig.dpi":600,"font.size":11,
                     "axes.edgecolor":"0.2","grid.color":"0.85"})
G_DARK, G_LIGHT = "0.25", "0.65"
def savefig(fig, stem):
    for e in ("png","pdf"): fig.savefig(FIG/(stem+"."+e), dpi=600, bbox_inches="tight")
    plt.close(fig)
def wilson(k,n,z=1.96):
    if n==0: return (np.nan,np.nan,np.nan)
    p=k/n; d=1+z*z/n; c=(p+z*z/(2*n))/d
    m=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/d
    return p, max(0,c-m), min(1,c+m)
def pct_ci(samples):
    s=np.asarray(samples); s=s[~np.isnan(s)]
    return float(np.mean(s)), float(np.percentile(s,2.5)), float(np.percentile(s,97.5))
OUT=[]
def rec(metric, p, lo, hi, n=None, method=""):
    OUT.append({"metric":metric,"estimate":round(p,4),"ci_lo":round(lo,4),
                "ci_hi":round(hi,4),"n":n,"method":method})

# ===================== CREDIT (01) =====================
df = pd.read_csv(DATA/"credit_german.csv"); df.columns=[c.strip().strip('"') for c in df.columns]
df["sex"]=np.where(df["personal_status_sex"].astype(str).str.contains("female"),"female","male")
y=(df["credit_risk"].astype(int)==1).astype(int)
X=df.drop(columns=["credit_risk","personal_status_sex"])
num=[c for c in X.select_dtypes("number").columns]; cat=[c for c in X.columns if c not in num and c!="sex"]
feat=[c for c in X.columns if c!="sex"]
def pre(cols): return ColumnTransformer([("num",StandardScaler(),[c for c in num if c in cols]),
    ("cat",OneHotEncoder(handle_unknown="ignore"),[c for c in cat if c in cols])])
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
proba=cross_val_predict(Pipeline([("pre",pre(feat)),("gb",GradientBoostingClassifier(random_state=SEED))]),
                        X[feat],y,cv=cv,method="predict_proba")[:,1]
pred=(proba>=0.5).astype(int); sexv=df["sex"].values
cred_rate={}
for g in ["female","male"]:
    k=int(pred[sexv==g].sum()); n=int((sexv==g).sum()); p,lo,hi=wilson(k,n)
    cred_rate[g]=(p,lo,hi,n); rec(f"credit_selection_rate_{g}",p,lo,hi,n,"Wilson")
# credit proxy AUC (LR predict female), bootstrap over sample
sx=(df["sex"]=="female").astype(int).values
proxy_p=cross_val_predict(Pipeline([("pre",pre(feat)),("lr",LogisticRegression(max_iter=2000))]),
                          X[feat],sx,cv=cv,method="predict_proba")[:,1]
def boot_auc(yt,sc):
    out=[]; idx=np.arange(len(yt))
    for _ in range(B):
        b=rng.choice(idx,len(idx),replace=True)
        if len(np.unique(yt[b]))<2: continue
        out.append(roc_auc_score(yt[b],sc[b]))
    return out
p,lo,hi=pct_ci(boot_auc(sx,proxy_p)); rec("credit_proxy_auc",p,lo,hi,len(sx),"bootstrap")
credit_proxy=(roc_auc_score(sx,proxy_p),lo,hi)

# ===================== HEALTH (02) =====================
dfh=pd.read_csv(DATA/"health_medcost.csv"); yh=dfh["charges"].values
def price_cv(features,catc):
    p=ColumnTransformer([("num",StandardScaler(),[c for c in features if c not in catc]),
        ("cat",OneHotEncoder(handle_unknown="ignore"),catc)])
    return cross_val_predict(Pipeline([("pre",p),("gb",GradientBoostingRegressor(random_state=SEED))]),
                             dfh[features],yh,cv=KFold(5,shuffle=True,random_state=SEED))
pred_blind=price_cv(["age","bmi","children","smoker","region"],["smoker","region"])
sexh=dfh["sex"].values
# bootstrap sex-blind premium ratio and DI ratio (resample individuals; predictions fixed)
thr=np.quantile(pred_blind,0.75); high=(pred_blind>=thr).astype(int)
idx=np.arange(len(sexh)); ratios=[]; dis=[]
for _ in range(B):
    b=rng.choice(idx,len(idx),replace=True)
    sb=sexh[b]; pb=pred_blind[b]; hb=high[b]
    mf=pb[sb=="female"].mean(); mm=pb[sb=="male"].mean()
    ratios.append(max(mf,mm)/min(mf,mm))
    hf=hb[sb=="female"].mean(); hm=hb[sb=="male"].mean()
    dis.append(min(hf,hm)/max(hf,hm))
p,lo,hi=pct_ci(ratios); rec("health_premium_ratio_sexblind",p,lo,hi,len(sexh),"bootstrap"); hr=(np.mean(ratios),lo,hi)
p,lo,hi=pct_ci(dis); rec("health_high_premium_DI_ratio",p,lo,hi,len(sexh),"bootstrap"); hdi=(np.mean(dis),lo,hi)
# health proxy AUC bootstrap
sxh=(dfh["sex"]=="male").astype(int).values
preh=ColumnTransformer([("num",StandardScaler(),["age","bmi","children"]),
    ("cat",OneHotEncoder(handle_unknown="ignore"),["smoker","region"])])
proxh=cross_val_predict(Pipeline([("pre",preh),("gb",GradientBoostingClassifier(random_state=SEED))]),
    dfh[["age","bmi","children","smoker","region"]],sxh,
    cv=StratifiedKFold(5,shuffle=True,random_state=SEED),method="predict_proba")[:,1]
p,lo,hi=pct_ci(boot_auc(sxh,proxh)); rec("health_proxy_auc",p,lo,hi,len(sxh),"bootstrap"); hproxy=(roc_auc_score(sxh,proxh),lo,hi)

# ===================== FRAUD (03) =====================
dff=pd.read_csv(DATA/"fraud_autoclaims.csv").replace("?",np.nan)
yf=(dff["fraud_reported"].astype(str).str.upper()=="Y").astype(int).values
sexf=dff["insured_sex"].astype(str).str.upper().values
drop=["fraud_reported","policy_number","policy_bind_date","insured_zip","incident_date",
      "incident_location","_c39","auto_model"]
Xf=dff.drop(columns=[c for c in drop if c in dff.columns])
numf=Xf.select_dtypes("number").columns.tolist(); catf=[c for c in Xf.columns if c not in numf]
Xf[catf]=Xf[catf].astype(str).fillna("missing")
pf=ColumnTransformer([("num",StandardScaler(),numf),("cat",OneHotEncoder(handle_unknown="ignore"),catf)])
probaf=cross_val_predict(Pipeline([("pre",pf),("gb",GradientBoostingClassifier(random_state=SEED))]),
    Xf,yf,cv=StratifiedKFold(5,shuffle=True,random_state=SEED),method="predict_proba")[:,1]
ths=np.linspace(0.05,0.95,91); t_acc=float(ths[int(np.argmax([accuracy_score(yf,(probaf>=t).astype(int)) for t in ths]))])
predf=(probaf>=t_acc).astype(int)
fpr_g={}
for g in ["MALE","FEMALE"]:
    m=(sexf==g); tn,fp,fn,tp=confusion_matrix(yf[m],predf[m],labels=[0,1]).ravel()
    k=int(fp); n=int(fp+tn); p,lo,hi=wilson(k,n); fpr_g[g]=(p,lo,hi,n)
    rec(f"fds_fpr_{g}",p,lo,hi,n,"Wilson (negatives)")
# restoration rate Wilson (restored / total FP)
fr=pd.read_csv(TAB/"fds_restoration.csv").iloc[0]
k=int(fr["fp_restored_by_review"]); n=int(fr["false_positives_total"]); p,lo,hi=wilson(k,n)
rec("fds_restoration_rate",p,lo,hi,n,"Wilson"); fds_rest=(p,lo,hi,n)

# ===================== LLM (04) — from committed per-case + offline reliance =====
per=pd.read_csv(TAB/"llm_explanation_per_case.csv")
faith_ci={}
for tier in ["weak","mid","strong"]:
    v=per[per.tier==tier]["faithfulness"].values; n=len(v)
    bs=[rng.choice(v,n,replace=True).mean() for _ in range(B)]
    p,lo,hi=pct_ci(bs); faith_ci[tier]=(v.mean(),lo,hi,n)
    rec(f"llm_faithfulness_{tier}",v.mean(),lo,hi,n,"bootstrap")
    rv=per[per.tier==tier]["recourse_valid"].values; k=int(round(rv.sum())); pp,llo,lhi=wilson(k,len(rv))
    rec(f"llm_recourse_valid_{tier}",pp,llo,lhi,len(rv),"Wilson")
# protected-reliance of the credit model over the committed case indices (no LLM)
cases=sorted(per["case"].unique().tolist())
Xc=df.drop(columns=["credit_risk","personal_status_sex"])
numc=Xc.select_dtypes("number").columns.tolist(); catc=[c for c in Xc.columns if c not in numc]
prec=ColumnTransformer([("num",StandardScaler(),numc),("cat",OneHotEncoder(handle_unknown="ignore"),catc)])
lrm=Pipeline([("pre",prec),("lr",LogisticRegression(max_iter=3000))]).fit(Xc,y)
coef=lrm.named_steps["lr"].coef_[0]; names=lrm.named_steps["pre"].get_feature_names_out()
PROT=["sex","gender","foreign_worker","foreign","age","marital"]
def ofeat(e):
    if e.startswith("num__"): return e[5:]
    if e.startswith("cat__"):
        r=e[5:]; cands=[c for c in catc if r.startswith(c+"_")]; return max(cands,key=len) if cands else r
    return e
MAP=[ofeat(n) for n in names]
def is_prot(f): return any(p in f.lower() for p in PROT)
def uses_prot(i,k=3):
    z=lrm.named_steps["pre"].transform(Xc.iloc[[i]]).ravel(); contrib=coef*z
    agg={}
    for f,c in zip(MAP,contrib): agg[f]=agg.get(f,0.0)+c
    top=[f for f,_ in sorted(agg.items(),key=lambda kv:kv[1])[:k]]
    return any(is_prot(f) for f in top)
rel=[uses_prot(int(i)) for i in cases]; k=int(sum(rel)); n=len(rel); p,lo,hi=wilson(k,n)
rec("credit_model_protected_reliance",p,lo,hi,n,"Wilson"); reliance=(p,lo,hi,n,k)

# ===================== REGENERATE BAR FIGURES WITH ERROR BARS =====================
def errbar_bar(stem, labels, ests, los, his, ylabel, ylim=(0,1), hline=None):
    fig,ax=plt.subplots(figsize=(4.2,3.2))
    x=np.arange(len(labels)); cols=[G_DARK,G_LIGHT]*((len(labels)+1)//2)
    yerr=np.array([[e-lo for e,lo in zip(ests,los)],[hi-e for e,hi in zip(ests,his)]])
    ax.bar(x,ests,color=cols[:len(labels)],edgecolor="black",width=0.6)
    ax.errorbar(x,ests,yerr=yerr,fmt="none",ecolor="0.1",capsize=4,lw=1.2)
    if hline is not None: ax.axhline(hline,color=G_LIGHT,ls="--",lw=1.2)
    ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylim(*ylim)
    ax.set_ylabel(ylabel); ax.set_xlabel("")
    savefig(fig,stem)
# credit selection rate
errbar_bar("fig_credit_selection_rate",["female","male"],
    [cred_rate["female"][0],cred_rate["male"][0]],
    [cred_rate["female"][1],cred_rate["male"][1]],
    [cred_rate["female"][2],cred_rate["male"][2]],"Approval (selection) rate")
# health proxy auc
errbar_bar("fig_health_proxy_auc",["sex-blind model"],[hproxy[0]],[hproxy[1]],[hproxy[2]],
    "Proxy-recovery AUC (sex)",hline=0.5)
# fds fpr by sex
errbar_bar("fig_fds_fpr_by_sex",["MALE","FEMALE"],
    [fpr_g["MALE"][0],fpr_g["FEMALE"][0]],[fpr_g["MALE"][1],fpr_g["FEMALE"][1]],
    [fpr_g["MALE"][2],fpr_g["FEMALE"][2]],"False positive rate",ylim=(0,0.30))
# llm faithfulness + recourse by tier with CIs
fig,ax=plt.subplots(figsize=(5.0,3.4)); x=np.arange(3); w=0.38
ftiers=["weak","mid","strong"]
fe=[faith_ci[t][0] for t in ftiers]; fl=[faith_ci[t][1] for t in ftiers]; fh=[faith_ci[t][2] for t in ftiers]
re_=[]; rl=[]; rh=[]
for t in ftiers:
    rv=per[per.tier==t]["recourse_valid"].values; p,lo,hi=wilson(int(round(rv.sum())),len(rv))
    re_.append(p); rl.append(lo); rh.append(hi)
ax.bar(x-w/2,fe,w,color=G_DARK,edgecolor="black",label="Faithfulness")
ax.errorbar(x-w/2,fe,yerr=[np.array(fe)-fl,np.array(fh)-fe],fmt="none",ecolor="0.1",capsize=3,lw=1.1)
ax.bar(x+w/2,re_,w,color=G_LIGHT,edgecolor="black",label="Recourse validity")
ax.errorbar(x+w/2,re_,yerr=[np.array(re_)-np.array(rl),np.array(rh)-np.array(re_)],fmt="none",ecolor="0.1",capsize=3,lw=1.1)
ax.set_xticks(x); ax.set_xticklabels(ftiers); ax.set_ylim(0,1)
ax.set_ylabel("Score"); ax.legend(frameon=False,fontsize=9)
savefig(fig,"fig_llm_faithfulness")

cis=pd.DataFrame(OUT); cis.to_csv(TAB/"bootstrap_cis.csv",index=False)
try: cis.to_latex(TAB/"bootstrap_cis.tex",index=False)
except Exception: pass
print(cis.to_string(index=False))
print("\nFAITHFULNESS (n=15/tier):", {t:round(faith_ci[t][0],3) for t in ftiers})
print("RELIANCE: %d/%d = %.3f CI[%.3f,%.3f]" % (reliance[4],reliance[3],reliance[0],reliance[1],reliance[2]))


                         metric  estimate  ci_lo  ci_hi    n             method
   credit_selection_rate_female    0.7548 0.7040 0.7994  310             Wilson
     credit_selection_rate_male    0.7855 0.7533 0.8145  690             Wilson
               credit_proxy_auc    0.6980 0.6617 0.7321 1000          bootstrap
  health_premium_ratio_sexblind    1.1566 1.0543 1.2650 1338          bootstrap
   health_high_premium_DI_ratio    0.7540 0.6242 0.9023 1338          bootstrap
               health_proxy_auc    0.4455 0.4166 0.4763 1338          bootstrap
                   fds_fpr_MALE    0.1725 0.1362 0.2161  342 Wilson (negatives)
                 fds_fpr_FEMALE    0.1630 0.1305 0.2018  411 Wilson (negatives)
           fds_restoration_rate    0.4762 0.3910 0.5628  126             Wilson
          llm_faithfulness_weak    0.1333 0.0222 0.2667   15          bootstrap
        llm_recourse_valid_weak    0.0000 0.0000 0.2039   15             Wilson
           llm_faithfulness_mid    0.333